# Predicción Mundial 2026 — el sistema completo en un notebook

Notebook **autocontenido**: el motor entero está en las celdas de la sección *Motor* y
no importa nada del repositorio. Se genera desde `src/` con
`scripts/build_notebook.py`; el código es el mismo que corren los scripts y los tests.

Estructura:

1. **Motor** — Elo y señales point-in-time, el panel de seis modelos, la mezcla
   calibrada, la simulación Monte Carlo y el backtest.
2. **Pronóstico** — corte al 8 de junio de 2026, con el inaugural México–Sudáfrica
   como hilo conductor.
3. **El examen** — backtest rolling-origin sobre 10 torneos, 482 partidos.

In [ ]:
from __future__ import annotations
import json, re, sys, time, warnings
from collections import defaultdict, deque
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import sparse
from scipy.optimize import minimize
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, PoissonRegressor
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"

## 1. Motor

### `src/config.py`

Parámetros del sistema — hoja v2 de la memoria técnica (Anexo B).

Un único lugar donde viven las perillas. Ningún módulo debe hardcodear
constantes que aparezcan aquí.

In [ ]:
# ---------------------------------------------------------------- Elo
ELO_K = 24.0
ELO_INITIAL = 1500.0
ELO_HOME_ADVANTAGE = 100.0  # Decisión 5.1: se probaron 55/80/100/120; ganó 100

# Peso por seriedad del torneo (Wtorneo en la ecuación de actualización)
ELO_TOURNAMENT_WEIGHTS = {
    "world_cup": 2.0,
    "continental": 1.6,
    "qualifier": 1.4,
    "nations_league": 1.2,
    "friendly": 0.75,
    "other": 1.0,
}

# ------------------------------------------------------- Entrenamiento
TRAIN_WINDOW_YEARS = 14  # ventana de entrenamiento para los modelos de señales
ML_RECENCY_HALFLIFE_YEARS = 8  # semivida del peso por recencia (la de 4 empeoró)
DC_RECENCY_HALFLIFE_YEARS = 3  # semivida del Dixon-Coles
DC_WINDOW_YEARS = 12  # ventana de datos del Dixon-Coles

# --------------------------------------------------------- Calibración
CALIBRATION_SPLIT = 0.30  # 30% cronológico final; NUNCA aleatorio (filtra futuro)

# ---------------------------------------------------------- Panelistas
LOGIT_C = 0.5  # regularización l2 del logit de 11 señales
ELO_LOGIT_C = 1.0  # regularización l2 del logit de 3 señales

XGB_PARAMS = dict(
    n_estimators=600,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=2.0,
    objective="multi:softprob",
    num_class=3,
    tree_method="hist",
)

RF_PARAMS = dict(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=10,
    n_jobs=-1,
)

DC_MAX_GOALS = 8  # truncamiento de la matriz de marcadores
DC_RHO_GRID = [-0.20, -0.15, -0.12, -0.10, -0.08, -0.05, -0.02, 0.0]

# ------------------------------------------------- Valor de plantilla
SQUAD_VALUE_WEIGHT = 0.45  # wV: extremo conservador del tramo plano 0.45-0.60

# ----------------------------------------------------------- Simulación
N_SIMULATIONS = 30_000
RANDOM_SEED = 2026  # semilla fija: la reproducibilidad es el argumento del repo

# ------------------------------------------------------- Modo en vivo
LIVE_MATCH_WEIGHT = 3.0  # triple peso vía sample_weight, no duplicando filas
LOW_CONFIDENCE_MIN_MATCHES = 18  # menos de 18 partidos en 24 meses => flag
LOW_CONFIDENCE_WINDOW_DAYS = 730

# ------------------------------------------------------------- Señales
FEATURE_NAMES = [
    "d_elo",
    "abs_d_elo",
    "true_home",
    "d_form",
    "d_gf",
    "d_ga",
    "d_gd",
    "d_sos",
    "d_wcexp",
    "same_confed",
    "d_recent",
]

ELO_FEATURE_NAMES = ["d_elo", "abs_d_elo", "true_home"]

FORM_WINDOW = 10  # últimos N partidos para forma, goles y dificultad de rivales

# ----------------------------------------------------------- Resultados
# Codificación del resultado: 0 = gana local, 1 = empate, 2 = gana visitante
OUTCOME_HOME, OUTCOME_DRAW, OUTCOME_AWAY = 0, 1, 2

### `src/data.py`

Carga y normalización de los datos.

Fuente única: results.csv (historial de partidos internacionales hasta el corte).
El fixture del Mundial 2026 vive en el mismo archivo y no tiene marcadores.

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = ROOT / "data"

def load_edition(name: str = "wc2026") -> dict:
    """Configuración de una edición: fechas, grupos, cuadro (ver editions/)."""
    return json.loads((ROOT / "editions" / f"{name}.json").read_text())

def edition_matches(df: pd.DataFrame, edition: dict) -> pd.DataFrame:
    """Los partidos de la edición presentes en el dataset, jugados o no."""
    return df[(df["tournament"] == edition["tournament"])
              & (df["date"] >= pd.Timestamp(edition["first_match"]))].copy()

# --------------------------------------------------------------------------
# Clasificación de torneos para el peso del Elo
# --------------------------------------------------------------------------

_CONTINENTAL = re.compile(
    r"UEFA Euro|Copa Am|African Cup|Africa Cup|AFC Asian Cup|Gold Cup|"
    r"Oceania Nations|CONCACAF Championship|Confederations Cup",
    re.I,
)
_QUALIFIER = re.compile(r"qualification|qualifier", re.I)

def tournament_weight_key(name: str) -> str:
    """Mapea el nombre del torneo a una de las categorías de peso del Elo."""
    if not isinstance(name, str):
        return "other"
    if _QUALIFIER.search(name):
        return "qualifier"
    if name.strip() == "FIFA World Cup":
        return "world_cup"
    if "Nations League" in name:
        return "nations_league"
    if _CONTINENTAL.search(name):
        return "continental"
    if "Friendly" in name:
        return "friendly"
    return "other"

# --------------------------------------------------------------------------
# Confederaciones (solo se necesitan las de los participantes y rivales
# frecuentes; el resto cae en "OTHER" y same_confed queda en 0)
# --------------------------------------------------------------------------

CONFEDERATIONS = {
    "UEFA": [
        "Spain", "France", "England", "Germany", "Portugal", "Netherlands",
        "Belgium", "Croatia", "Italy", "Switzerland", "Austria", "Denmark",
        "Sweden", "Norway", "Poland", "Ukraine", "Czech Republic", "Scotland",
        "Wales", "Republic of Ireland", "Northern Ireland", "Serbia", "Turkey",
        "Greece", "Romania", "Hungary", "Russia", "Slovakia", "Slovenia",
        "Bosnia and Herzegovina", "Iceland", "Finland", "Albania", "Bulgaria",
        "North Macedonia", "Montenegro", "Georgia", "Israel", "Kosovo",
        "Belarus", "Armenia", "Azerbaijan", "Kazakhstan", "Cyprus", "Estonia",
        "Latvia", "Lithuania", "Luxembourg", "Malta", "Moldova", "Faroe Islands",
        "Gibraltar", "Andorra", "San Marino", "Liechtenstein",
    ],
    "CONMEBOL": [
        "Brazil", "Argentina", "Uruguay", "Colombia", "Chile", "Peru",
        "Ecuador", "Paraguay", "Venezuela", "Bolivia",
    ],
    "CONCACAF": [
        "Mexico", "United States", "Canada", "Costa Rica", "Panama", "Jamaica",
        "Honduras", "Haiti", "Curaçao", "Trinidad and Tobago", "El Salvador",
        "Guatemala", "Nicaragua", "Suriname", "Cuba", "Martinique",
        "Guadeloupe", "Bermuda", "Grenada", "Saint Kitts and Nevis",
        "Dominican Republic", "Belize", "Antigua and Barbuda", "Barbados",
        "Guyana", "Puerto Rico", "Aruba", "Saint Lucia",
    ],
    "CAF": [
        "Morocco", "Senegal", "Egypt", "Nigeria", "Algeria", "Tunisia",
        "Ivory Coast", "Cameroon", "Ghana", "South Africa", "Mali",
        "DR Congo", "Burkina Faso", "Cape Verde", "Guinea", "Zambia",
        "Angola", "Uganda", "Benin", "Gabon", "Kenya", "Mozambique",
        "Madagascar", "Congo", "Sudan", "Tanzania", "Zimbabwe", "Namibia",
        "Equatorial Guinea", "Libya", "Togo", "Mauritania", "Guinea-Bissau",
        "Sierra Leone", "Niger", "Malawi", "Comoros", "Ethiopia", "Rwanda",
        "Botswana", "Burundi", "Liberia", "Central African Republic", "Chad",
        "Lesotho", "Eswatini", "Gambia", "Somalia", "South Sudan",
    ],
    "AFC": [
        "Japan", "South Korea", "Iran", "Australia", "Saudi Arabia", "Qatar",
        "Iraq", "Uzbekistan", "Jordan", "United Arab Emirates", "China PR",
        "China", "Oman", "Bahrain", "Syria", "Lebanon", "Palestine", "Kuwait",
        "Vietnam", "Thailand", "Indonesia", "Malaysia", "Philippines",
        "Singapore", "India", "Tajikistan", "Kyrgyzstan", "Turkmenistan",
        "North Korea", "Hong Kong", "Chinese Taipei", "Myanmar", "Bangladesh",
        "Nepal", "Maldives", "Afghanistan", "Yemen", "Cambodia", "Laos",
    ],
    "OFC": [
        "New Zealand", "New Caledonia", "Fiji", "Tahiti", "Papua New Guinea",
        "Solomon Islands", "Vanuatu", "Samoa", "Tonga", "American Samoa",
        "Cook Islands",
    ],
}

TEAM_TO_CONFED = {
    team: confed for confed, teams in CONFEDERATIONS.items() for team in teams
}

def confederation(team: str) -> str:
    return TEAM_TO_CONFED.get(team, "OTHER")

# --------------------------------------------------------------------------
# Carga
# --------------------------------------------------------------------------

def load_results(path: Path | str | None = None) -> pd.DataFrame:
    """Carga results.csv normalizado y ordenado cronológicamente."""
    path = Path(path) if path else DATA_DIR / "results.csv"
    df = pd.read_csv(path, parse_dates=["date"])

    df["neutral"] = df["neutral"].astype(str).str.upper().isin(["TRUE", "1"])
    df["played"] = df["home_score"].notna() & df["away_score"].notna()
    df["w_key"] = df["tournament"].map(tournament_weight_key)

    # Localía real: el local juega en su país y la cancha no es neutral
    df["true_home"] = (~df["neutral"]) & (df["country"] == df["home_team"])

    df["outcome"] = np.select(
        [df["home_score"] > df["away_score"], df["home_score"] == df["away_score"]],
        [OUTCOME_HOME, OUTCOME_DRAW],
        default=OUTCOME_AWAY,
    ).astype(float)
    df.loc[~df["played"], "outcome"] = np.nan

    df = df.sort_values("date", kind="mergesort").reset_index(drop=True)
    df["match_id"] = df.index
    return df

def load_groups(path: Path | str | None = None) -> pd.DataFrame:
    """Grupos del sorteo oficial del Mundial 2026."""
    path = Path(path) if path else DATA_DIR / "groups_2026.csv"
    return pd.read_csv(path)

def load_squad_values(path: Path | str | None = None) -> pd.DataFrame | None:
    """
    Valores de plantilla. Devuelve None si el archivo no tiene valores usables,
    en cuyo caso el panelista de valor se desactiva (wV = 0) y el pipeline
    sigue corriendo con los cinco panelistas históricos.
    """
    path = Path(path) if path else DATA_DIR / "squad_2026.csv"
    if not path.exists():
        return None
    df = pd.read_csv(path)
    if "value_gbp_m" not in df.columns or df["value_gbp_m"].notna().sum() < 40:
        return None
    df = df.dropna(subset=["value_gbp_m"]).copy()
    log_v = np.log(df["value_gbp_m"])
    df["z_log_value"] = (log_v - log_v.mean()) / log_v.std(ddof=0)
    return df[["team", "value_gbp_m", "z_log_value"]]

def world_cup_2026(df: pd.DataFrame) -> pd.DataFrame:
    return edition_matches(df, load_edition("wc2026"))

def cut_at(df: pd.DataFrame, date: str | pd.Timestamp) -> pd.DataFrame:
    """
    Corta el dataset a una fecha, dejando los partidos posteriores como
    fixture (marcador vacío). Es la operación que permite reproducir el
    pronóstico pre-torneo: cut_at(df, '2026-06-08').
    """
    date = pd.Timestamp(date)
    out = df.copy()
    future = out["date"] > date
    out.loc[future, ["home_score", "away_score", "outcome"]] = np.nan
    out.loc[future, "played"] = False
    return out

### `src/engine.py`

Elo y señales, en una sola pasada cronológica.

La regla sagrada del proyecto vive aquí y es estructural, no una auditoría:
para cada partido se ANOTAN primero las señales con el estado acumulado hasta
ese momento, y solo DESPUÉS se procesa el resultado. Es físicamente imposible
que una señal conozca su propio partido.

In [ ]:
from collections import defaultdict, deque

import numpy as np
import pandas as pd

def _mean(values, default=0.0):
    return sum(values) / len(values) if values else default

def build_features(df: pd.DataFrame, return_state: bool = False):
    """
    Recorre los partidos en orden cronológico y devuelve el DataFrame original
    con las once señales, los ratings previos y las banderas de confianza.

    Los partidos sin marcador (fixture) reciben señales igual que los demás,
    pero no actualizan el estado: no hay resultado que aprender.
    """
    df = df.sort_values("date", kind="mergesort").reset_index(drop=True)

    elo: dict[str, float] = defaultdict(lambda: ELO_INITIAL)
    form: dict[str, deque] = defaultdict(lambda: deque(maxlen=FORM_WINDOW))
    wc_played: dict[str, int] = defaultdict(int)
    recent_dates: dict[str, deque] = defaultdict(deque)

    n = len(df)
    cols = {name: np.zeros(n) for name in FEATURE_NAMES}
    elo_home = np.zeros(n)
    elo_away = np.zeros(n)
    recent_h = np.zeros(n, dtype=int)
    recent_a = np.zeros(n, dtype=int)

    window = pd.Timedelta(days=LOW_CONFIDENCE_WINDOW_DAYS)

    home_arr = df["home_team"].to_numpy()
    away_arr = df["away_team"].to_numpy()
    date_arr = df["date"].to_numpy()
    th_arr = df["true_home"].to_numpy()
    hs_arr = df["home_score"].to_numpy()
    as_arr = df["away_score"].to_numpy()
    wk_arr = df["w_key"].to_numpy()
    played_arr = df["played"].to_numpy()

    for i in range(n):
        h, a = home_arr[i], away_arr[i]
        date = pd.Timestamp(date_arr[i])
        true_home = bool(th_arr[i])

        # -- purga de la ventana de actividad (solo mira hacia atrás) --------
        for team in (h, a):
            dq = recent_dates[team]
            while dq and (date - dq[0]) > window:
                dq.popleft()

        rh, ra = elo[h], elo[a]
        elo_home[i], elo_away[i] = rh, ra

        d = rh + (ELO_HOME_ADVANTAGE if true_home else 0.0) - ra

        fh, fa = form[h], form[a]
        cols["d_elo"][i] = d / 400.0
        cols["abs_d_elo"][i] = abs(d) / 400.0
        cols["true_home"][i] = 1.0 if true_home else 0.0
        cols["d_form"][i] = _mean([r[0] for r in fh]) - _mean([r[0] for r in fa])
        cols["d_gf"][i] = _mean([r[1] for r in fh]) - _mean([r[1] for r in fa])
        cols["d_ga"][i] = _mean([r[2] for r in fh]) - _mean([r[2] for r in fa])
        cols["d_gd"][i] = (
            _mean([r[1] - r[2] for r in fh]) - _mean([r[1] - r[2] for r in fa])
        )
        cols["d_sos"][i] = (
            _mean([r[3] for r in fh], ELO_INITIAL)
            - _mean([r[3] for r in fa], ELO_INITIAL)
        ) / 400.0
        cols["d_wcexp"][i] = np.log1p(wc_played[h]) - np.log1p(wc_played[a])
        cols["same_confed"][i] = (
            1.0 if confederation(h) == confederation(a) != "OTHER" else 0.0
        )
        nh, na = len(recent_dates[h]), len(recent_dates[a])
        recent_h[i], recent_a[i] = nh, na
        cols["d_recent"][i] = float(nh - na)

        # -- a partir de aquí se consume el resultado ------------------------
        if not played_arr[i]:
            continue

        gh, ga_ = float(hs_arr[i]), float(as_arr[i])
        s_home = 1.0 if gh > ga_ else (0.5 if gh == ga_ else 0.0)
        expected = 1.0 / (1.0 + 10.0 ** (-d / 400.0))

        w = ELO_TOURNAMENT_WEIGHTS.get(wk_arr[i], 1.0)
        margin = 1.0 + np.log1p(abs(gh - ga_))
        delta = ELO_K * w * margin * (s_home - expected)

        elo[h] = rh + delta
        elo[a] = ra - delta

        pts_h = 3.0 if gh > ga_ else (1.0 if gh == ga_ else 0.0)
        pts_a = 3.0 if ga_ > gh else (1.0 if gh == ga_ else 0.0)
        form[h].append((pts_h, gh, ga_, ra))
        form[a].append((pts_a, ga_, gh, rh))

        if wk_arr[i] == "world_cup":
            wc_played[h] += 1
            wc_played[a] += 1

        recent_dates[h].append(date)
        recent_dates[a].append(date)

    out = df.copy()
    for name, values in cols.items():
        out[name] = values
    out["elo_home"] = elo_home
    out["elo_away"] = elo_away
    out["recent_home"] = recent_h
    out["recent_away"] = recent_a
    out["low_confidence"] = (recent_h < LOW_CONFIDENCE_MIN_MATCHES) | (
        recent_a < LOW_CONFIDENCE_MIN_MATCHES
    )
    if not return_state:
        return out

    # Estado final de cada selección: lo que necesita un cruce hipotético de
    # eliminación directa para describirse con las once señales completas y no
    # con ceros. Sale de los mismos acumuladores, así que es point-in-time por
    # construcción igual que el resto.
    state = {}
    for team in set(home_arr) | set(away_arr):
        rec = form[team]
        state[team] = {
            "elo": elo[team],
            "form": _mean([r[0] for r in rec]),
            "gf": _mean([r[1] for r in rec]),
            "ga": _mean([r[2] for r in rec]),
            "gd": _mean([r[1] - r[2] for r in rec]),
            "sos": _mean([r[3] for r in rec], ELO_INITIAL),
            "wcexp": wc_played[team],
            "recent": len(recent_dates[team]),
        }
    return out, state

def final_ratings(featured: pd.DataFrame, as_of: pd.Timestamp | str) -> pd.Series:
    """
    Rating Elo de cada selección tras el último partido jugado hasta `as_of`.
    Se obtiene de las columnas elo_home/elo_away más la actualización
    del último partido, evitando una segunda pasada.
    """
    as_of = pd.Timestamp(as_of)
    sub = featured[(featured["date"] <= as_of) & featured["played"]]

    ratings: dict[str, float] = {}
    for _, row in sub.iterrows():
        d = (
            row["elo_home"]
            + (ELO_HOME_ADVANTAGE if row["true_home"] else 0.0)
            - row["elo_away"]
        )
        gh, ga = row["home_score"], row["away_score"]
        s = 1.0 if gh > ga else (0.5 if gh == ga else 0.0)
        expected = 1.0 / (1.0 + 10.0 ** (-d / 400.0))
        w = ELO_TOURNAMENT_WEIGHTS.get(row["w_key"], 1.0)
        delta = ELO_K * w * (1.0 + np.log1p(abs(gh - ga))) * (s - expected)
        ratings[row["home_team"]] = row["elo_home"] + delta
        ratings[row["away_team"]] = row["elo_away"] - delta

    return pd.Series(ratings).sort_values(ascending=False)

### `src/models.py`

El panel de modelos.

Seis maneras de opinar sobre el mismo partido, elegidas para equivocarse de
formas distintas. Todos exponen la misma interfaz: fit(train) / predict(X)
devolviendo una matriz (n, 3) de probabilidades [local, empate, visitante].

In [ ]:
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, PoissonRegressor
from sklearn.preprocessing import StandardScaler

def recency_weights(dates: pd.Series, as_of: pd.Timestamp, halflife_years: float):
    """Peso 2^(-antigüedad/semivida): el modelo olvida gradualmente."""
    age_years = (pd.Timestamp(as_of) - dates).dt.days / 365.25
    return np.power(2.0, -age_years / halflife_years)

# --------------------------------------------------------------------------
# Modelos de señales
# --------------------------------------------------------------------------

class _SignalModel:
    """Base para los panelistas que consumen la matriz de once señales."""

    features = FEATURE_NAMES

    def __init__(self):
        self.scaler = StandardScaler()
        self.model = None

    def fit(self, train: pd.DataFrame, as_of, weights=None):
        X = self.scaler.fit_transform(train[self.features].to_numpy())
        y = train["outcome"].to_numpy().astype(int)
        if weights is None:
            weights = recency_weights(train["date"], as_of, ML_RECENCY_HALFLIFE_YEARS)
        self._fit_model(X, y, np.asarray(weights))
        return self

    def predict(self, X_df: pd.DataFrame) -> np.ndarray:
        X = self.scaler.transform(X_df[self.features].to_numpy())
        return self.model.predict_proba(X)

class Logit(_SignalModel):
    name = "logit"

    def _fit_model(self, X, y, w):
        self.model = LogisticRegression(C=LOGIT_C, max_iter=2000)
        self.model.fit(X, y, sample_weight=w)

class EloLogit(_SignalModel):
    name = "elo_logit"
    features = ELO_FEATURE_NAMES

    def _fit_model(self, X, y, w):
        self.model = LogisticRegression(C=ELO_LOGIT_C, max_iter=2000)
        self.model.fit(X, y, sample_weight=w)

class XGB(_SignalModel):
    name = "xgboost"

    def _fit_model(self, X, y, w):
        from xgboost import XGBClassifier

        params = {k: v for k, v in XGB_PARAMS.items() if k not in ("objective", "num_class")}
        self.model = XGBClassifier(**params, objective="multi:softprob", num_class=3)
        self.model.fit(X, y, sample_weight=w, verbose=False)

class RF(_SignalModel):
    name = "random_forest"

    def _fit_model(self, X, y, w):
        self.model = RandomForestClassifier(**RF_PARAMS, random_state=0)
        self.model.fit(X, y, sample_weight=w)

# --------------------------------------------------------------------------
# Dixon-Coles: el panelista que habla en goles
# --------------------------------------------------------------------------

class DixonColes:
    """
    Poisson por equipo (ataque propio contra defensa rival) con la corrección
    de Dixon y Coles (1997) para marcadores bajos, y decaimiento temporal.

    Los parámetros de ataque y defensa se ajustan con una regresión de Poisson
    sobre un diseño disperso: cada partido aporta dos observaciones.
    """

    name = "dixon_coles"

    def __init__(self, max_goals: int = DC_MAX_GOALS):
        self.max_goals = max_goals
        self.rho = -0.1
        self.teams: list[str] = []

    # ---- ajuste ----------------------------------------------------------
    def fit(self, train: pd.DataFrame, as_of, weights=None):
        as_of = pd.Timestamp(as_of)
        window_start = as_of - pd.Timedelta(days=int(365.25 * DC_WINDOW_YEARS))
        sub = train[train["date"] >= window_start]
        if len(sub) < 500:
            sub = train

        self.teams = sorted(set(sub["home_team"]) | set(sub["away_team"]))
        idx = {t: i for i, t in enumerate(self.teams)}
        n_teams = len(self.teams)
        n = len(sub)

        # dos filas por partido: goles del local y goles del visitante
        rows = np.repeat(np.arange(2 * n), 3)
        attack = np.concatenate(
            [sub["home_team"].map(idx).to_numpy(), sub["away_team"].map(idx).to_numpy()]
        )
        defense = np.concatenate(
            [sub["away_team"].map(idx).to_numpy(), sub["home_team"].map(idx).to_numpy()]
        )
        home_flag = np.concatenate([sub["true_home"].to_numpy().astype(float),
                                    np.zeros(n)])
        cols = np.column_stack([attack, n_teams + defense,
                                np.full(2 * n, 2 * n_teams)]).ravel()
        vals = np.column_stack([np.ones(2 * n), np.ones(2 * n), home_flag]).ravel()

        X = sparse.csr_matrix(
            (vals, (rows, cols)), shape=(2 * n, 2 * n_teams + 1)
        )
        y = np.concatenate([sub["home_score"].to_numpy(), sub["away_score"].to_numpy()])

        w = recency_weights(sub["date"], as_of, DC_RECENCY_HALFLIFE_YEARS)
        w = np.concatenate([w, w])

        glm = PoissonRegressor(alpha=1e-4, max_iter=800, fit_intercept=True)
        glm.fit(X, y, sample_weight=w)

        self.intercept = glm.intercept_
        self.attack = glm.coef_[:n_teams]
        self.defense = glm.coef_[n_teams:2 * n_teams]
        self.gamma = glm.coef_[2 * n_teams]
        self._idx = idx

        self._fit_rho(sub)
        return self

    def _fit_rho(self, sub: pd.DataFrame):
        """rho por malla sobre el 15% cronológico final del entrenamiento."""
        tail = sub.iloc[int(len(sub) * 0.85):]
        if len(tail) < 100:
            return
        lam_h, lam_a = self._lambdas(tail)
        gh = tail["home_score"].to_numpy().astype(int)
        ga = tail["away_score"].to_numpy().astype(int)

        best, best_ll = self.rho, -np.inf
        for rho in DC_RHO_GRID:
            tau = self._tau(gh, ga, lam_h, lam_a, rho)
            if np.any(tau <= 0):
                continue
            ll = np.sum(np.log(tau))
            if ll > best_ll:
                best_ll, best = ll, rho
        self.rho = best

    # ---- predicción ------------------------------------------------------
    def _lambdas(self, X_df: pd.DataFrame):
        default_a = float(np.mean(self.attack))
        default_d = float(np.mean(self.defense))
        ah = X_df["home_team"].map(lambda t: self.attack[self._idx[t]]
                                   if t in self._idx else default_a).to_numpy()
        dh = X_df["home_team"].map(lambda t: self.defense[self._idx[t]]
                                   if t in self._idx else default_d).to_numpy()
        aa = X_df["away_team"].map(lambda t: self.attack[self._idx[t]]
                                   if t in self._idx else default_a).to_numpy()
        da = X_df["away_team"].map(lambda t: self.defense[self._idx[t]]
                                   if t in self._idx else default_d).to_numpy()
        home = X_df["true_home"].to_numpy().astype(float)
        lam_h = np.exp(self.intercept + ah + da + self.gamma * home)
        lam_a = np.exp(self.intercept + aa + dh)
        return lam_h, lam_a

    @staticmethod
    def _tau(gh, ga, lam_h, lam_a, rho):
        """Corrección de Dixon-Coles sobre las celdas (0,0),(0,1),(1,0),(1,1)."""
        tau = np.ones_like(lam_h, dtype=float)
        m00 = (gh == 0) & (ga == 0)
        m01 = (gh == 0) & (ga == 1)
        m10 = (gh == 1) & (ga == 0)
        m11 = (gh == 1) & (ga == 1)
        tau[m00] = 1 - lam_h[m00] * lam_a[m00] * rho
        tau[m01] = 1 + lam_h[m01] * rho
        tau[m10] = 1 + lam_a[m10] * rho
        tau[m11] = 1 - rho
        return np.clip(tau, 1e-9, None)

    def score_matrix(self, X_df: pd.DataFrame) -> np.ndarray:
        """Matriz (n, max_goals+1, max_goals+1) de probabilidades de marcador."""
        lam_h, lam_a = self._lambdas(X_df)
        k = np.arange(self.max_goals + 1)
        log_fact = np.cumsum(np.concatenate([[0.0], np.log(np.arange(1, self.max_goals + 1))]))

        lp_h = (-lam_h[:, None] + k[None, :] * np.log(lam_h[:, None]) - log_fact[None, :])
        lp_a = (-lam_a[:, None] + k[None, :] * np.log(lam_a[:, None]) - log_fact[None, :])
        mat = np.exp(lp_h[:, :, None] + lp_a[:, None, :])

        rho = self.rho
        mat[:, 0, 0] *= 1 - lam_h * lam_a * rho
        mat[:, 0, 1] *= 1 + lam_h * rho
        mat[:, 1, 0] *= 1 + lam_a * rho
        mat[:, 1, 1] *= 1 - rho
        mat = np.clip(mat, 1e-12, None)
        return mat / mat.sum(axis=(1, 2), keepdims=True)

    def predict(self, X_df: pd.DataFrame) -> np.ndarray:
        mat = self.score_matrix(X_df)
        iu = np.triu_indices(self.max_goals + 1, k=1)
        il = np.tril_indices(self.max_goals + 1, k=-1)
        away = mat[:, iu[0], iu[1]].sum(axis=1)
        home = mat[:, il[0], il[1]].sum(axis=1)
        draw = np.einsum("ijj->i", mat)
        out = np.column_stack([home, draw, away])
        return out / out.sum(axis=1, keepdims=True)

# --------------------------------------------------------------------------
# Valor de plantilla
# --------------------------------------------------------------------------

class SquadValue:
    """
    Un logit sobre una sola señal: la diferencia de valor de plantilla
    estandarizado. Casi trivial, y valioso justo por eso: aporta información
    que ningún otro panelista tiene.
    """

    name = "squad_value"

    def __init__(self, values: pd.DataFrame):
        self.z = dict(zip(values["team"], values["z_log_value"]))
        self.model = None

    def _signal(self, X_df: pd.DataFrame) -> np.ndarray:
        vh = X_df["home_team"].map(lambda t: self.z.get(t, 0.0)).to_numpy()
        va = X_df["away_team"].map(lambda t: self.z.get(t, 0.0)).to_numpy()
        return np.column_stack([vh - va, X_df["true_home"].to_numpy().astype(float)])

    def fit(self, train: pd.DataFrame, as_of, weights=None):
        mask = train["home_team"].isin(self.z) & train["away_team"].isin(self.z)
        sub = train[mask]
        X = self._signal(sub)
        y = sub["outcome"].to_numpy().astype(int)
        w = recency_weights(sub["date"], as_of, ML_RECENCY_HALFLIFE_YEARS)
        self.model = LogisticRegression(max_iter=1000)
        self.model.fit(X, y, sample_weight=w)
        return self

    def predict(self, X_df: pd.DataFrame) -> np.ndarray:
        return self.model.predict_proba(self._signal(X_df))

PANEL = [EloLogit, Logit, XGB, RF, DixonColes]

### `src/ensemble.py`

De seis opiniones a una: mezcla, calibración y métricas.

Los pesos no se asignan, se aprenden sobre el 30% cronológicamente más
reciente del entrenamiento. Reservar partidos al azar dejaría colarse al
futuro y violaría la regla sagrada.

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

# --------------------------------------------------------------------------
# Métricas
# --------------------------------------------------------------------------

def log_loss(p: np.ndarray, y: np.ndarray) -> float:
    p = np.clip(p, 1e-15, 1.0)
    return float(-np.mean(np.log(p[np.arange(len(y)), y])))

def rps(p: np.ndarray, y: np.ndarray) -> float:
    """Ranked Probability Score: respeta el orden victoria-empate-derrota."""
    obs = np.zeros_like(p)
    obs[np.arange(len(y)), y] = 1.0
    cp, co = np.cumsum(p, axis=1), np.cumsum(obs, axis=1)
    return float(np.mean(np.sum((cp - co) ** 2, axis=1) / (p.shape[1] - 1)))

def brier(p: np.ndarray, y: np.ndarray) -> float:
    obs = np.zeros_like(p)
    obs[np.arange(len(y)), y] = 1.0
    return float(np.mean(np.sum((p - obs) ** 2, axis=1)))

def accuracy(p: np.ndarray, y: np.ndarray) -> float:
    return float(np.mean(np.argmax(p, axis=1) == y))

def ece(p: np.ndarray, y: np.ndarray, n_bins: int = 10) -> float:
    """
    Error esperado de calibración sobre TODAS las probabilidades emitidas
    (las tres por partido), no solo la máxima. Es la definición que usa el
    proyecto para su segunda vara.
    """
    obs = np.zeros_like(p)
    obs[np.arange(len(y)), y] = 1.0
    conf, hit = p.ravel(), obs.ravel()
    edges = np.linspace(0, 1, n_bins + 1)
    total, n = 0.0, len(conf)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() == 0:
            continue
        total += m.sum() / n * abs(conf[m].mean() - hit[m].mean())
    return float(total)

def all_metrics(p: np.ndarray, y: np.ndarray) -> dict:
    return {
        "log_loss": log_loss(p, y),
        "rps": rps(p, y),
        "brier": brier(p, y),
        "ece": ece(p, y),
        "accuracy": accuracy(p, y),
    }

# --------------------------------------------------------------------------
# Mezcla calibrada
# --------------------------------------------------------------------------

def _blend(probs: list[np.ndarray], w: np.ndarray, T: float) -> np.ndarray:
    stacked = np.stack(probs, axis=0)
    mixed = np.tensordot(w, stacked, axes=(0, 0))
    logits = np.log(np.clip(mixed, 1e-15, None)) / T
    logits -= logits.max(axis=1, keepdims=True)
    e = np.exp(logits)
    return e / e.sum(axis=1, keepdims=True)

class Ensemble:
    """Panel de cinco modelos históricos + calibración por temperatura."""

    def __init__(self, panel=None):
        self.panel_classes = panel or PANEL
        self.models = []
        self.weights = None
        self.temperature = 1.0

    def fit(self, train: pd.DataFrame, as_of):
        train = train.sort_values("date", kind="mergesort")
        split = int(len(train) * (1 - CALIBRATION_SPLIT))
        fit_part, cal_part = train.iloc[:split], train.iloc[split:]

        # Los panelistas se ajustan con el tramo inicial; los pesos y la
        # temperatura se aprenden sobre el tramo reservado, que ninguno vio.
        self.models = [cls().fit(fit_part, as_of) for cls in self.panel_classes]

        probs = [m.predict(cal_part) for m in self.models]
        y = cal_part["outcome"].to_numpy().astype(int)
        self.weights, self.temperature = self._optimize(probs, y)

        # Reajuste final sobre todo el entrenamiento, con los pesos ya fijados
        self.models = [cls().fit(train, as_of) for cls in self.panel_classes]
        return self

    @staticmethod
    def _optimize(probs, y):
        k = len(probs)

        def objective(theta):
            w = np.exp(theta[:k])
            w /= w.sum()
            T = np.exp(theta[k])
            return log_loss(_blend(probs, w, T), y)

        best, best_val = None, np.inf
        for seed in (np.zeros(k + 1), np.concatenate([np.linspace(-1, 1, k), [0.0]])):
            res = minimize(objective, seed, method="Nelder-Mead",
                           options={"maxiter": 4000, "xatol": 1e-4, "fatol": 1e-6})
            if res.fun < best_val:
                best_val, best = res.fun, res.x

        w = np.exp(best[:k])
        w /= w.sum()
        w[w < 0.005] = 0.0  # poda: pesos despreciables se anulan explícitamente
        w /= w.sum()
        return w, float(np.exp(best[k]))

    def predict(self, X_df: pd.DataFrame) -> np.ndarray:
        probs = [m.predict(X_df) for m in self.models]
        return _blend(probs, self.weights, self.temperature)

    @property
    def dixon_coles(self):
        for m in self.models:
            if m.name == "dixon_coles":
                return m
        return None

    def report_weights(self) -> dict:
        return {m.name: float(w) for m, w in zip(self.models, self.weights)}

class Forecaster:
    """
    El sistema completo: ensemble histórico + panelista de valor de plantilla,
    que solo se mezcla para los partidos del Mundial 2026.
    """

    def __init__(self, squad_values: pd.DataFrame | None = None,
                 value_weight: float = SQUAD_VALUE_WEIGHT):
        self.ensemble = Ensemble()
        self.squad_values = squad_values
        self.value_weight = value_weight if squad_values is not None else 0.0
        self.value_model = None

    def fit(self, train: pd.DataFrame, as_of):
        self.ensemble.fit(train, as_of)
        if self.squad_values is not None:
            self.value_model = SquadValue(self.squad_values).fit(train, as_of)
        return self

    def predict(self, X_df: pd.DataFrame, use_value: bool = False) -> np.ndarray:
        p = self.ensemble.predict(X_df)
        if use_value and self.value_model is not None and self.value_weight > 0:
            pv = self.value_model.predict(X_df)
            p = (1 - self.value_weight) * p + self.value_weight * pv
            p /= p.sum(axis=1, keepdims=True)
        return p

### `src/simulate.py`

Del partido al torneo: simulación Monte Carlo.

Saber el 71% de un partido no responde "¿quién será campeón?". Entre medio hay
reglas, terceros y cruces. La solución honesta: jugar el torneo, muchas veces.

En cada réplica se sortea primero el RESULTADO con las ternas del ensemble
(el mejor juez del 1X2) y después un MARCADOR del Dixon-Coles compatible con
ese resultado. Cada componente hace lo que mejor sabe.

In [ ]:
import numpy as np
import pandas as pd

# --------------------------------------------------------------------------
# Estructura del cuadro (Reglamento FIFA 2026, Annex C)
#
# Recuperada de los cruces reales de dieciseisavos del torneo. La estructura
# del cuadro es información pública anterior al torneo (sale del sorteo), así
# que usarla no introduce información del futuro: solo recupera la tabla.
#
# LIMITACIÓN DOCUMENTADA: la tabla oficial asigna los 8 mejores terceros a
# ranuras específicas según DE QUÉ GRUPOS provienen, con una tabla de
# combinaciones. Aquí se usa una asignación por orden de mérito, que coincide
# con la oficial en la mayoría de los casos pero no en todos. Es la principal
# fuente de error de las probabilidades por ronda.
# --------------------------------------------------------------------------

R32_SLOTS = [
    ("A2", "B2"), ("C1", "F2"), ("E1", "3rd"), ("F1", "C2"),
    ("E2", "I2"), ("I1", "3rd"), ("A1", "3rd"), ("L1", "3rd"),
    ("G1", "3rd"), ("D1", "3rd"), ("H1", "J2"), ("K2", "L2"),
    ("B1", "3rd"), ("D2", "G2"), ("J1", "H2"), ("K1", "3rd"),
]

GROUPS = list("ABCDEFGHIJKL")

def _rank_group(teams, pts, gd, gf):
    """Ordena un grupo por puntos, diferencia de gol y goles a favor."""
    return sorted(teams, key=lambda t: (-pts[t], -gd[t], -gf[t], t))

class TournamentSimulator:
    def __init__(self, fixtures: pd.DataFrame, probs: np.ndarray,
                 score_matrix: np.ndarray, groups: pd.DataFrame,
                 n_sims: int = N_SIMULATIONS, seed: int = RANDOM_SEED,
                 r32_slots=None):
        """
        fixtures: los 72 partidos de grupos, en orden
        probs: (72, 3) probabilidades 1X2 del ensemble
        score_matrix: (72, G+1, G+1) matriz de marcadores del Dixon-Coles
        """
        self.fx = fixtures.reset_index(drop=True)
        self.r32_slots = [tuple(s) for s in (r32_slots or R32_SLOTS)]
        self.probs = probs
        self.n_sims = n_sims
        self.rng = np.random.default_rng(seed)

        self.group_of = dict(zip(groups["team"], groups["group"]))
        self.teams_in = {g: list(groups.loc[groups["group"] == g, "team"])
                         for g in GROUPS}

        self._prepare_score_draws(score_matrix)

    # ---- preparación: marcadores condicionados al resultado --------------
    def _prepare_score_draws(self, mat):
        """
        Para cada partido y cada resultado posible, la distribución de
        marcadores compatible, normalizada. Se muestrea de ahí.
        """
        n, k, _ = mat.shape
        ii, jj = np.meshgrid(np.arange(k), np.arange(k), indexing="ij")
        masks = [ii > jj, ii == jj, ii < jj]  # local, empate, visitante

        self.score_choices = []
        for outcome, mask in enumerate(masks):
            flat_idx = np.flatnonzero(mask.ravel())
            w = mat.reshape(n, -1)[:, flat_idx]
            w = w / w.sum(axis=1, keepdims=True)
            self.score_choices.append((flat_idx, np.cumsum(w, axis=1), k))

    def _draw_scores(self, outcomes):
        """outcomes: (n_sims, 72) -> goles local y visitante."""
        n_sims, n_matches = outcomes.shape
        gh = np.zeros((n_sims, n_matches), dtype=np.int16)
        ga = np.zeros((n_sims, n_matches), dtype=np.int16)
        u = self.rng.random((n_sims, n_matches))

        for outcome, (flat_idx, cumw, k) in enumerate(self.score_choices):
            sel = outcomes == outcome
            if not sel.any():
                continue
            sim_i, match_i = np.nonzero(sel)
            pos = np.array([
                np.searchsorted(cumw[m], u[s, m]) for s, m in zip(sim_i, match_i)
            ])
            pos = np.clip(pos, 0, cumw.shape[1] - 1)
            flat = flat_idx[pos]
            gh[sim_i, match_i] = flat // k
            ga[sim_i, match_i] = flat % k
        return gh, ga

    # ---- una réplica completa -------------------------------------------
    def run(self, advance_prob_fn, verbose: bool = True):
        """
        advance_prob_fn(team_a, team_b) -> probabilidad de que A avance
        (en cancha neutral, ya incluyendo la aproximación de penales).
        """
        n = self.n_sims
        n_g = len(self.fx)
        u = self.rng.random((n, n_g))
        cum = np.cumsum(self.probs, axis=1)
        outcomes = (u[:, :, None] > cum[None, :, :]).sum(axis=2).astype(np.int8)
        gh, ga = self._draw_scores(outcomes)

        home = self.fx["home_team"].to_numpy()
        away = self.fx["away_team"].to_numpy()

        counters = {
            "r32": {}, "r16": {}, "qf": {}, "sf": {}, "final": {}, "champion": {},
            "group_winner": {}, "advance": {}, "top2": {},
        }

        for s in range(n):
            bundle = self._group_tables(home, away, gh[s], ga[s])
            standings = bundle[0]
            qualified, slots = self._qualifiers(bundle)

            for t in qualified:
                counters["advance"][t] = counters["advance"].get(t, 0) + 1
            for g in GROUPS:
                w = standings[g][0]
                counters["group_winner"][w] = counters["group_winner"].get(w, 0) + 1
                for t2 in standings[g][:2]:
                    counters["top2"][t2] = counters["top2"].get(t2, 0) + 1

            self._knockout(slots, advance_prob_fn, counters)

            if verbose and (s + 1) % 5000 == 0:
                print(f"    {s + 1:,}/{n:,} réplicas")

        return {k: {t: c / n for t, c in v.items()} for k, v in counters.items()}

    def _group_tables(self, home, away, gh, ga):
        pts, gd, gf = {}, {}, {}
        for g in GROUPS:
            for t in self.teams_in[g]:
                pts[t] = gd[t] = gf[t] = 0

        for m in range(len(home)):
            h, a, x, y = home[m], away[m], int(gh[m]), int(ga[m])
            gd[h] += x - y
            gd[a] += y - x
            gf[h] += x
            gf[a] += y
            if x > y:
                pts[h] += 3
            elif x < y:
                pts[a] += 3
            else:
                pts[h] += 1
                pts[a] += 1

        return {g: _rank_group(self.teams_in[g], pts, gd, gf) for g in GROUPS}, pts, gd, gf

    def _qualifiers(self, standings_bundle):
        standings, pts, gd, gf = standings_bundle
        thirds = [(standings[g][2], g) for g in GROUPS]
        thirds.sort(key=lambda x: (-pts[x[0]], -gd[x[0]], -gf[x[0]], x[0]))
        best_thirds = [t for t, _ in thirds[:8]]

        slots = {}
        for g in GROUPS:
            slots[f"{g}1"] = standings[g][0]
            slots[f"{g}2"] = standings[g][1]

        qualified = list(slots.values()) + best_thirds
        return qualified, (slots, best_thirds)

    def _knockout(self, slot_bundle, adv_fn, counters):
        slots, best_thirds = slot_bundle
        third_iter = iter(best_thirds)

        bracket = []
        for a, b in self.r32_slots:
            ta = next(third_iter) if a == "3rd" else slots[a]
            tb = next(third_iter) if b == "3rd" else slots[b]
            bracket.append((ta, tb))

        for t in [x for pair in bracket for x in pair]:
            counters["r32"][t] = counters["r32"].get(t, 0) + 1

        for round_name in ("r16", "qf", "sf", "final", "champion"):
            winners = []
            for ta, tb in bracket:
                p = adv_fn(ta, tb)
                winners.append(ta if self.rng.random() < p else tb)
            for t in winners:
                counters[round_name][t] = counters[round_name].get(t, 0) + 1
            if len(winners) < 2:
                break
            bracket = [(winners[i], winners[i + 1]) for i in range(0, len(winners), 2)]

def make_advance_fn(forecaster, featured_row_builder):
    """
    Devuelve una función (a, b) -> P(a avanza), usando el modelo en cancha
    neutral con la aproximación de penales: ADV = pW + 0.5 * pD.
    """
    cache: dict[tuple[str, str], float] = {}

    def adv(a: str, b: str) -> float:
        key = (a, b)
        if key in cache:
            return cache[key]
        row = featured_row_builder(a, b)
        p = forecaster.predict(row, use_value=True)[0]
        value = float(p[0] + 0.5 * p[1])
        cache[key] = value
        cache[(b, a)] = 1.0 - value
        return value

    return adv

### `src/backtest.py`

El examen: backtest rolling-origin sobre torneos reales.

Para cada torneo, se para el reloj la víspera de su primer partido, se entrena
solo con lo anterior, se predicen sus partidos y se califica contra la
historia. Promedia por torneo para que el Mundial (64 partidos) no ahogue a la
Copa América (26).

    python src/backtest.py                # los 10 torneos de la memoria
    python src/backtest.py --quick        # solo los 3 más cortos, para probar

In [ ]:
import argparse
import json
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

TOURNAMENTS = [
    ("FIFA World Cup", 2014), ("FIFA World Cup", 2018), ("FIFA World Cup", 2022),
    ("UEFA Euro", 2016), ("UEFA Euro", 2021), ("UEFA Euro", 2024),
    ("Copa América", 2019), ("Copa América", 2021), ("Copa América", 2024),
    ("AFC Asian Cup", 2019),
]

def run_fold(df: pd.DataFrame, name: str, year: int) -> dict:
    target = df[(df["tournament"] == name) & (df["date"].dt.year == year)]
    cut = target["date"].min() - pd.Timedelta(days=1)

    feat = build_features(cut_at(df, cut))
    train = feat[feat["played"]
                 & (feat["date"] <= cut)
                 & (feat["date"] >= cut - pd.Timedelta(days=365.25 * TRAIN_WINDOW_YEARS))]
    ens = Ensemble().fit(train, cut)

    test = feat.loc[target.index]
    y = target["outcome"].to_numpy().astype(int)
    out = {"tournament": f"{name} {year}", "n": len(test), "cut": str(cut.date())}
    out["ensemble"] = all_metrics(ens.predict(test), y)
    out["panel"] = {m.name: all_metrics(m.predict(test), y) for m in ens.models}
    out["weights"] = ens.report_weights()
    return out

## 2. Pronóstico pre-torneo

**Corte de datos: 8 de junio de 2026.** Nada posterior entra al modelo. Todo lo que
sigue es lo que un analista podía calcular la víspera del torneo.

In [ ]:
ed = load_edition("wc2026")
CUT = pd.Timestamp(ed["cut"])
df = cut_at(load_results(), CUT)
groups = load_groups(ROOT / ed["groups"])
squad = load_squad_values(ROOT / ed["squad_values"])
print(f"{ed['name']} · {int(df.played.sum()):,} partidos hasta {CUT.date()}")

### Elo y las once señales

Una sola pasada cronológica: para cada partido se anotan primero las señales con el
estado acumulado, y solo después se procesa el resultado. Ninguna señal conoce su propio
partido, por construcción.

In [ ]:
feat, state = build_features(df, return_state=True)
ratings = final_ratings(feat, CUT)
ratings.head(10).round(0).to_frame("Elo")

In [ ]:
opener = edition_matches(feat, ed).iloc[0]
d = opener.elo_home + 100 * opener.true_home - opener.elo_away
print(f"{opener.home_team} {opener.elo_home:.0f} (+100 localía) vs {opener.away_team} {opener.elo_away:.0f}")
print(f"d = {d:.0f}  →  expectativa Elo {1/(1+10**(-d/400)):.2f}   (la memoria: d=437, 0.92)")
opener[FEATURE_NAMES].round(3)

### El panel y la mezcla

Cinco modelos históricos, pesos aprendidos sobre el 30% cronológico final, calibración
por temperatura. El sexto —valor de plantilla— entra solo para los partidos de la
edición, con peso 0.45.

In [ ]:
train = feat[feat.played & (feat.date <= CUT) & (feat.date >= CUT - pd.Timedelta(days=365.25*TRAIN_WINDOW_YEARS))]
fc = Forecaster(squad).fit(train, CUT)
pd.Series(fc.ensemble.report_weights(), name="peso").round(3).to_frame().T.assign(T=round(fc.ensemble.temperature, 3))

In [ ]:
fixtures = edition_matches(feat, ed)
fixtures = fixtures[fixtures.date > CUT].head(ed["group_matches"]).reset_index(drop=True)
probs = fc.predict(fixtures, use_value=True)
dc = fc.ensemble.dixon_coles
scores = dc.score_matrix(fixtures)
lam_h, lam_a = dc._lambdas(fixtures)

pred = fixtures[["home_team", "away_team"]].copy()
pred[["p_home", "p_draw", "p_away"]] = (probs * 100).round(0)
best = scores.reshape(len(scores), -1).argmax(1); k = scores.shape[1]
pred["marcador"] = [f"{b//k}-{b%k}" for b in best]
pred["xG"] = [f"{a:.2f} / {b:.2f}" for a, b in zip(lam_h, lam_a)]
pred.head(6)

El inaugural publicado el 10 de junio fue **71 / 19 / 10, marcador 1–0, xG 1.67 / 0.62**.

### 30,000 Mundiales

Se sortea primero el resultado con las ternas del ensemble, y después un marcador del
Dixon–Coles compatible. Reglas FIFA, mejores terceros, cuadro oficial, y penales como
moneda cargada al favorito.

In [ ]:
default = {"elo": 1500., "form": 0., "gf": 0., "ga": 0., "gd": 0., "sos": 1500., "wcexp": 0, "recent": 0}
def neutral_row(a, b):
    sa, sb = state.get(a, default), state.get(b, default); d = sa["elo"] - sb["elo"]
    return pd.DataFrame([{"home_team": a, "away_team": b, "true_home": 0., "d_elo": d/400, "abs_d_elo": abs(d)/400,
        "d_form": sa["form"]-sb["form"], "d_gf": sa["gf"]-sb["gf"], "d_ga": sa["ga"]-sb["ga"], "d_gd": sa["gd"]-sb["gd"],
        "d_sos": (sa["sos"]-sb["sos"])/400, "d_wcexp": float(np.log1p(sa["wcexp"])-np.log1p(sb["wcexp"])),
        "same_confed": float(confederation(a) == confederation(b) != "OTHER"), "d_recent": float(sa["recent"]-sb["recent"])}])

sim = TournamentSimulator(fixtures, probs, scores, groups, n_sims=30_000, seed=RANDOM_SEED, r32_slots=ed["r32_slots"])
res = sim.run(make_advance_fn(fc, neutral_row), verbose=False)
tab = pd.DataFrame({r: res[r] for r in ["top2", "r16", "qf", "sf", "final", "champion"]}).fillna(0)
tab.columns = ["Top 2", "Octavos", "Cuartos", "Semis", "Final", "Campeón"]
assert abs(tab["Campeón"].sum() - 1) < 1e-9
(tab.sort_values("Campeón", ascending=False).head(12) * 100).round(1)

In [ ]:
top = tab["Campeón"].sort_values(ascending=False).head(10) * 100
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(top.index[::-1], top.values[::-1], color=["#2a9d3f" if t == "Spain" else "#9aa5b1" for t in top.index[::-1]])
for i, v in enumerate(top.values[::-1]): ax.text(v + 0.2, i, f"{v:.1f}%", va="center", fontsize=9)
ax.set_xlabel("P(campeón) %"); ax.set_title(f"{ed['name']} — favoritos al título, corte {CUT.date()}")
ax.spines[["top", "right"]].set_visible(False); fig.tight_layout()
(ROOT / "docs/figures").mkdir(parents=True, exist_ok=True)
fig.savefig(ROOT / "docs/figures/title_odds.png", dpi=150); plt.show()

## 3. El examen: backtest de 10 torneos

Para cada torneo se para el reloj la víspera de su primer partido, se entrena solo con
lo anterior, se predicen sus partidos y se califica. Promedio por torneo. Unos dos
minutos.

In [ ]:
full = load_results()
folds, t0 = [], time.time()
for name, year in TOURNAMENTS:
    r = run_fold(full, name, year); folds.append(r)
    print(f"  {r['tournament']:<20} {r['n']:3d} partidos  log-loss {r['ensemble']['log_loss']:.3f}  acierto {r['ensemble']['accuracy']*100:.0f}%   ({time.time()-t0:.0f}s)")

In [ ]:
rows = [{"model": "ensemble", **{k: np.mean([r["ensemble"][k] for r in folds]) for k in ("log_loss", "rps", "brier", "ece", "accuracy")}}]
for mname in folds[0]["panel"]:
    rows.append({"model": mname, **{k: np.mean([r["panel"][mname][k] for r in folds]) for k in ("log_loss", "rps", "brier", "ece", "accuracy")}})
print(f"{len(folds)} torneos, {sum(r['n'] for r in folds)} partidos · azar uniforme: log-loss {np.log(3):.3f}")
print("publicado en la memoria: ensemble 0.943, acierto 56.4%")
pd.DataFrame(rows).sort_values("log_loss").round(3)